# House price model: how it was built

This notebook walks through the training steps behind the API, using the same code the API runs (`app/models/price_model.py`):

1. Load and look at the data
2. Train the old baseline (linear regression, 6 fields) to see where it fails
3. Train the final model (gradient boosting, 9 fields + ratio features)
4. Check it is stable with cross-validation
5. See which inputs matter most

To create the model file the API uses, run `python train.py` from the repository root. This notebook doesn't save anything.

In [1]:
import sys

sys.path.insert(0, "..")  # run from notebooks/: make the app package importable

import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

from app.models import price_model

## 1. The data

California Housing (1990 US census). Each row is a **block group** (a small neighborhood), not one house. The target is `median_house_value` in USD.

In [2]:
raw = pd.read_csv(price_model.DATA_URL)
print("Rows, columns:", raw.shape)
print("Missing total_bedrooms:", raw["total_bedrooms"].isna().sum())
print("Rows at the $500,001 cap:", (raw["median_house_value"] == 500_001).sum())
raw.head()

Rows, columns: (20640, 10)
Missing total_bedrooms: 207
Rows at the $500,001 cap: 965


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
raw["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

`ISLAND` has only 5 rows, so predictions for islands are less reliable.

## 2. Train/test split

`split_data()` drops the 207 rows with no `total_bedrooms` and holds out 20% of the rows for testing. The split is the same on every run (`random_state=42`).

In [4]:
X_train, X_test, y_train, y_test = price_model.split_data()
print("Train rows:", len(X_train), " Test rows:", len(X_test))

Train rows: 16346  Test rows: 4087


## 3. Baseline: linear regression on 6 fields

This was the first version of the API. It ignores location.

In [5]:
OLD_FIELDS = ["housing_median_age", "total_rooms", "total_bedrooms",
              "population", "households", "median_income"]

baseline = LinearRegression().fit(X_train[OLD_FIELDS], y_train)
base_preds = baseline.predict(X_test[OLD_FIELDS])
print(f"Baseline test MAE: ${mean_absolute_error(y_test, base_preds):,.0f}")
print(f"Baseline test R^2: {r2_score(y_test, base_preds):.3f}")
print(f"Negative predictions: {(base_preds < 0).sum()}")

Baseline test MAE: $56,642
Baseline test R^2: 0.571
Negative predictions: 4


## 4. Final model: gradient boosting on 9 fields

`build_pipeline()` chains three steps:

1. `add_ratio_features` adds rooms per household, bedrooms per room and people per household.
2. One-hot encoding turns `ocean_proximity` into numbers.
3. `HistGradientBoostingRegressor` learns the price from all of it, including longitude and latitude.

In [6]:
model = price_model.build_pipeline().fit(X_train, y_train)
metrics = price_model.evaluate(model)
print(f"Final test MAE: ${metrics['mae']:,.0f}")
print(f"Final test R^2: {metrics['r2']:.3f}")

Final test MAE: $29,362
Final test R^2: 0.852


## 5. Is the error stable?

5-fold cross-validation trains 5 models on different parts of the training set. A small spread means the result isn't luck of one split.

In [7]:
cv_mae = -cross_val_score(price_model.build_pipeline(), X_train, y_train,
                          cv=5, scoring="neg_mean_absolute_error")
print("MAE per fold:", [f"${m:,.0f}" for m in cv_mae])
print(f"Mean ${cv_mae.mean():,.0f}, spread +/- ${cv_mae.std():,.0f}")

MAE per fold: ['$28,224', '$31,472', '$29,793', '$29,926', '$29,721']
Mean $29,827, spread +/- $1,029


## 6. Which inputs matter?

Permutation importance shuffles one input at a time and measures how much worse the test error gets.

In [8]:
imp = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42,
                             scoring="neg_mean_absolute_error")
(pd.Series(imp.importances_mean, index=X_test.columns)
   .sort_values(ascending=False)
   .map(lambda v: f"${v:,.0f} worse MAE when shuffled"))

latitude              $39,258 worse MAE when shuffled
longitude             $38,687 worse MAE when shuffled
median_income         $32,004 worse MAE when shuffled
total_rooms           $24,264 worse MAE when shuffled
population            $19,208 worse MAE when shuffled
households            $16,693 worse MAE when shuffled
ocean_proximity       $13,140 worse MAE when shuffled
total_bedrooms         $6,522 worse MAE when shuffled
housing_median_age     $4,204 worse MAE when shuffled
dtype: str

## Summary

| Model | Fields | Test MAE | Test R² |
|-------|--------|----------|---------|
| Linear regression (old) | 6 | see section 3 | see section 3 |
| Gradient boosting (current) | 9 + 3 ratios | see section 4 | see section 4 |

Location (`longitude`, `latitude`) and `median_income` drive most of the improvement. Run `python train.py` to train and save the model the API uses.